In [ ]:

# Assume `coffee_data` has columns "latitude" and "longitude"
coords = coffee_data[['latitude', 'longitude']].values

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA


# 1) Compute Euclidean distance matrix
D = squareform(pdist(coords))

# 2) Choose a truncation threshold
#    A common choice is the distance of the k-th nearest neighbor (e.g. k=4)
k_nn = 4
sorted_dists = np.sort(D, axis=1)
threshold = np.mean(sorted_dists[:, k_nn])  # average 4th‐NN distance

# 3) Build a binary spatial weights matrix W
W = (D <= threshold).astype(int)
np.fill_diagonal(W, 0)  # zero diagonal

# 4) Center W to obtain spatial filtering matrix S = H W H
n = W.shape[0]
I = np.eye(n)
one = np.ones((n, n)) / n
H = I - one
S = H.dot(W).dot(H)

# 5) Eigen‐decompose S
eigvals, eigvecs = np.linalg.eigh(S)

# 6) Select Moran’s eigenvector maps (MEMs) with positive eigenvalues
pos_mask = eigvals > 0
mem = eigvecs[:, pos_mask]
pos_vals = eigvals[pos_mask]

# 7) Optionally choose a subset (e.g. top 10) by descending eigenvalue
top_k = 10
order = np.argsort(pos_vals)[::-1][:top_k]
mem_k = mem[:, order]

# 8) Append MEM columns to your DataFrame for modeling
mem_cols = [f'MEM_{i+1}' for i in range(mem_k.shape[1])]
coffee_data[mem_cols] = mem_k

print("Added MEM columns:", mem_cols)
